In [1]:
import numpy as np
import jax
import torch
import torch_geometric

/home/tam/miniconda3/envs/egnn-3.11/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
flag = [True]
def remove_self_loop(data):
    if flag[0]: 
        print("transform")
        flag[0]=False
    edge_index = data.edge_index
    edge_attr = data.edge_attr
    new_edge_index, new_edge_attr = torch_geometric.utils.remove_self_loops(edge_index, edge_attr)
    data.edge_index = new_edge_index
    data.edge_attr = new_edge_attr
    return data

In [3]:
qm9_dataset = torch_geometric.datasets.QM9(root='data/QM9', pre_transform=remove_self_loop)

In [4]:
qm9_dataset

QM9(130831)

In [5]:
qm9_dataset[0].edge_attr

tensor([[1., 0., 0., 0.],
        [1., 0., 0., 0.],
        [1., 0., 0., 0.],
        [1., 0., 0., 0.],
        [1., 0., 0., 0.],
        [1., 0., 0., 0.],
        [1., 0., 0., 0.],
        [1., 0., 0., 0.]])

In [5]:
qm9_dataset.shuffle()

QM9(130831)

In [6]:
qm9_dataset[:10]

QM9(10)

In [10]:
qm9_dataset[10].z, qm9_dataset[10].x

(tensor([6, 6, 8, 1, 1, 1, 1]),
 tensor([[0., 1., 0., 0., 0., 6., 0., 0., 0., 0., 3.],
         [0., 1., 0., 0., 0., 6., 0., 0., 0., 0., 1.],
         [0., 0., 0., 1., 0., 8., 0., 0., 0., 0., 0.],
         [1., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0.],
         [1., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0.],
         [1., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0.],
         [1., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0.]]))

In [11]:
qm9_dataset[10].x.int()

tensor([[0, 1, 0, 0, 0, 6, 0, 0, 0, 0, 3],
        [0, 1, 0, 0, 0, 6, 0, 0, 0, 0, 1],
        [0, 0, 0, 1, 0, 8, 0, 0, 0, 0, 0],
        [1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0],
        [1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0],
        [1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0],
        [1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0]], dtype=torch.int32)

In [7]:
qm9_dataset[10]

Data(x=[7, 11], edge_index=[2, 12], edge_attr=[12, 4], y=[1, 19], pos=[7, 3], z=[7], smiles='[H]C(=O)C([H])([H])[H]', name='gdb_11', idx=[1])

In [ ]:
qm9_dataset

In [14]:
from qm9_dataloader import z_to_15_features
def transfrom_data(data):
    z = data.z
    data.feature = z_to_15_features(z)
    return data

In [ ]:
qm9_dataset

In [23]:
qm9_dataset_transform = [transfrom_data(x) for x in qm9_dataset ]

In [27]:
qm9_dataset_transform[8:11]

[Data(x=[7, 11], edge_index=[2, 12], edge_attr=[12, 4], y=[1, 19], pos=[7, 3], z=[7], smiles='[H]C#CC([H])([H])[H]', name='gdb_9', idx=[1], feature=[7, 15]),
 Data(x=[6, 11], edge_index=[2, 10], edge_attr=[10, 4], y=[1, 19], pos=[6, 3], z=[6], smiles='[H]C([H])([H])C#N', name='gdb_10', idx=[1], feature=[6, 15]),
 Data(x=[7, 11], edge_index=[2, 12], edge_attr=[12, 4], y=[1, 19], pos=[7, 3], z=[7], smiles='[H]C(=O)C([H])([H])[H]', name='gdb_11', idx=[1], feature=[7, 15])]

In [29]:
torch.func.vmap

<function torch.func.vmap(func: 'Callable[_P, _R]', in_dims: 'in_dims_t' = 0, out_dims: 'out_dims_t' = 0, randomness: 'str' = 'error', *, chunk_size: 'int | None' = None) -> 'Callable[_P, _R]'>

In [42]:
def cal_relative_matrix(coord, edge_index):
    # neighbor_coord = []
    # print(vmap_coor(edge_index[0], edge_index[1]))
    # for src, target in edge_index.T:
    #     dist = (coord[target] - coord[src])
    #     print(dist.shape)
    #     neighbor_coord.append(dist)
    # print(neighbor_coord)
    # return torch.vstack(neighbor_coord)
    vmap_coor = torch.func.vmap(lambda s, t : coord[t] - coord[s])
    return vmap_coor(edge_index[0], edge_index[1])

cal_relative_matrix(qm9_dataset_transform[0].pos, qm9_dataset_transform[0].edge_index)

tensor([[ 0.0149, -1.0918, -0.0060],
        [ 1.0244,  0.3780, -0.0077],
        [-0.5281,  0.3617, -0.8846],
        [-0.5111,  0.3521,  0.8984],
        [-0.0149,  1.0918,  0.0060],
        [-1.0244, -0.3780,  0.0077],
        [ 0.5281, -0.3617,  0.8846],
        [ 0.5111, -0.3521, -0.8984]])

In [45]:
cal_relative_matrix(qm9_dataset_transform[10].pos, qm9_dataset_transform[10].edge_index).norm(keepdim = True, dim = -1)

tensor([[1.5076],
        [1.0911],
        [1.0967],
        [1.0968],
        [1.5076],
        [1.2036],
        [1.1150],
        [1.2036],
        [1.0911],
        [1.0967],
        [1.0968],
        [1.1150]])

In [1]:
from qm9_dataloader import get_train_val_test
train, val, test = get_train_val_test()

/home/tam/miniconda3/envs/egnn-3.11/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Warning seed is constant
Train: 100000, Val: 17747, Test: 13084


In [4]:
sample = next(iter(train))
sample

DataBatch(x=[30, 11], edge_index=[2, 64], edge_attr=[64, 4], y=[2, 19], pos=[30, 3], z=[30], smiles=[2], name=[2], idx=[2], feature=[30, 15], relatix_matrix=[64, 3], batch=[30], ptr=[3])

In [5]:
sample.batch

tensor([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1])

In [8]:
import jax.numpy as jnp

In [10]:
batch = jnp.array(sample.batch)

In [12]:
num_nodes = jnp.zeros((batch[-1] + 1,), dtype=int).at[batch].add(jnp.ones_like(batch)).reshape(-1,1)
num_nodes

Array([[18],
       [12]], dtype=int32)

In [13]:
num_nodes[batch]

Array([[18],
       [18],
       [18],
       [18],
       [18],
       [18],
       [18],
       [18],
       [18],
       [18],
       [18],
       [18],
       [18],
       [18],
       [18],
       [18],
       [18],
       [18],
       [12],
       [12],
       [12],
       [12],
       [12],
       [12],
       [12],
       [12],
       [12],
       [12],
       [12],
       [12]], dtype=int32)

In [30]:
qm9_dataset.shuffle()[10]

Data(x=[20, 11], edge_index=[2, 42], edge_attr=[42, 4], y=[1, 19], pos=[20, 3], z=[20], smiles='[H]C([H])([H])C(=O)[C@]1([H])[N@H+]2[C@@]([H])(C([H])([H])[H])[C@@]2([H])C1([H])[H]', name='gdb_80584', idx=[1])

In [24]:
torch_geometric.utils.smiles.from_smiles('[H]C(=O)C([H])([H])[H]',with_hydrogen=True)

Data(x=[7, 9], edge_index=[2, 12], edge_attr=[12, 3], smiles='[H]C(=O)C([H])([H])[H]')

In [7]:
max([i.num_nodes for i in qm9_dataset])

29

In [10]:
import jax
import jax.numpy as jnp

In [13]:
loop_index = jnp.arange(0, 10).reshape((1, -1))
print(loop_index)
loop_index = jnp.repeat(loop_index, repeats=2, axis=0)
loop_index

[[0 1 2 3 4 5 6 7 8 9]]


Array([[0, 1, 2, 3, 4, 5, 6, 7, 8, 9],
       [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]], dtype=int32)

In [16]:
loop_index

Array([[0, 1, 2, 3, 4, 5, 6, 7, 8, 9],
       [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]], dtype=int32)

In [20]:
a = jnp.array([float("inf")])
a

Array([inf], dtype=float32)

In [21]:
a == float("inf")

Array([ True], dtype=bool)